# 02 — Data Cleaning

## Objective

This notebook cleans and prepares the healthcare datasets for analysis and modeling.

The cleaning process will:

- Load the original raw datasets without modifying them.
- Create separate working copies for cleaning.
- Remove unnecessary whitespace from text labels.
- Fill the grouped workforce category labels.
- Convert the year columns from strings to numeric values.
- Separate records that use different measurement units.
- Validate and save the cleaned datasets.

## 1. Setup and Load Raw Data

Load the original datasets and create separate working copies for cleaning.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"

inpatients_path = raw_dir / "Inpatients_in_Health_Sectors_Hospitals__KSA.csv"
workforce_path = raw_dir / "PHC_Health_Workforce__MoH___Last_5_Years.csv"

inpatients_raw = pd.read_csv(inpatients_path)
workforce_raw = pd.read_csv(workforce_path)

inpatients_clean = inpatients_raw.copy()
workforce_clean = workforce_raw.copy()

print("Inpatients working copy:", inpatients_clean.shape)
print("Workforce working copy:", workforce_clean.shape)

Inpatients working copy: (5, 6)
Workforce working copy: (24, 7)


## 2. Clean Text Labels

Remove leading and trailing whitespace from text columns.

In [2]:
inpatients_clean["Sector"] = inpatients_clean["Sector"].str.strip()

workforce_clean["Nationality"] = workforce_clean["Nationality"].str.strip()
workforce_clean["Category"] = workforce_clean["Category"].str.strip()

print("Cleaned sector labels:")
print(inpatients_clean["Sector"].tolist())

print("\nCleaned nationality labels:")
print(workforce_clean["Nationality"].unique())

print("\nCleaned category labels:")
print(workforce_clean["Category"].dropna().tolist())

Cleaned sector labels:
['MoH', 'Other Governmental Sector', 'Private Sector', 'Total', 'Average no. of admissions per 100 persons']

Cleaned nationality labels:
<StringArray>
['Saudi', 'Non-Saudi', 'Total']
Length: 3, dtype: str

Cleaned category labels:
['Physicians', 'Dentists', 'Total Physicians and Dentists', 'Nurses', 'Midwives', 'Total Nurses and Midwives', 'Pharmacists', 'Allied health Personnel']


## 3. Fill Grouped Workforce Categories

Propagate each workforce category label to its related nationality rows.

In [3]:
print("Missing categories before:", workforce_clean["Category"].isna().sum())

workforce_clean["Category"] = workforce_clean["Category"].ffill()

print("Missing categories after:", workforce_clean["Category"].isna().sum())

display(workforce_clean[["Category", "Nationality"]])

Missing categories before: 16
Missing categories after: 0


,Category,Nationality
0,Physicians,Saudi
1,Physicians,Non-Saudi
2,Physicians,Total
3,Dentists,Saudi
4,Dentists,Non-Saudi
5,Dentists,Total
6,Total Physicians and Dentists,Saudi
7,Total Physicians and Dentists,Non-Saudi
8,Total Physicians and Dentists,Total
9,Nurses,Saudi


## 4. Detect and Convert Year Columns

Automatically detect year columns, remove thousands separators, and convert their values from strings to numeric values.

In [4]:
inpatient_year_columns = sorted([
    column
    for column in inpatients_clean.columns
    if (
        isinstance(column, str)
        and column.endswith("G")
        and column[:-1].isdigit()
    )
])

workforce_year_columns = sorted([
    column
    for column in workforce_clean.columns
    if (
        isinstance(column, str)
        and column.endswith("G")
        and column[:-1].isdigit()
    )
])

if not inpatient_year_columns:
    raise ValueError("No inpatient year columns were detected.")

if not workforce_year_columns:
    raise ValueError("No workforce year columns were detected.")

for column in inpatient_year_columns:
    inpatients_clean[column] = pd.to_numeric(
        inpatients_clean[column].str.replace(",", "", regex=False)
    )

for column in workforce_year_columns:
    workforce_clean[column] = pd.to_numeric(
        workforce_clean[column].str.replace(",", "", regex=False)
    )

print("Detected inpatient years:", inpatient_year_columns)
print("Detected workforce years:", workforce_year_columns)

print("\nInpatients year column types:")
print(inpatients_clean[inpatient_year_columns].dtypes)

print("\nWorkforce year column types:")
print(workforce_clean[workforce_year_columns].dtypes)

Detected inpatient years: ['2020G', '2021G', '2022G', '2023G', '2024G']
Detected workforce years: ['2020G', '2021G', '2022G', '2023G', '2024G']

Inpatients year column types:
2020G    float64
2021G    float64
2022G    float64
2023G    float64
2024G    float64
dtype: object

Workforce year column types:
2020G    int64
2021G    int64
2022G    int64
2023G    int64
2024G    int64
dtype: object


## 5. Separate Measurement Units

Separate inpatient counts from the admissions rate and assign the appropriate numeric type to each dataset.

In [5]:
rate_label = "Average no. of admissions per 100 persons"

inpatients_counts = inpatients_clean[
    inpatients_clean["Sector"] != rate_label
].copy()

inpatients_rate = inpatients_clean[
    inpatients_clean["Sector"] == rate_label
].copy()

inpatients_counts[inpatient_year_columns] = (
    inpatients_counts[inpatient_year_columns].astype("Int64")
)

inpatients_rate[inpatient_year_columns] = (
    inpatients_rate[inpatient_year_columns].astype("float64")
)

print("Inpatient count rows:", inpatients_counts.shape)
print("Admission rate rows:", inpatients_rate.shape)

print("\nInpatient count types:")
print(inpatients_counts[inpatient_year_columns].dtypes)

print("\nAdmission rate types:")
print(inpatients_rate[inpatient_year_columns].dtypes)

display(inpatients_counts)
display(inpatients_rate)

Inpatient count rows: (4, 6)
Admission rate rows: (1, 6)

Inpatient count types:
2020G    Int64
2021G    Int64
2022G    Int64
2023G    Int64
2024G    Int64
dtype: object

Admission rate types:
2020G    float64
2021G    float64
2022G    float64
2023G    float64
2024G    float64
dtype: object


,2020G,2021G,2022G,2023G,2024G,Sector
0,1173910,1376787,1389950,1418237,1876723,MoH
1,553214,620735,544306,629879,671669,Other Governmental Sector
2,1119649,1103230,768204,1783708,1081942,Private Sector
3,2846773,3100752,2702460,3831824,3630334,Total


,2020G,2021G,2022G,2023G,2024G,Sector
4,9.0,10.1,8.4,11.3,10.3,Average no. of admissions per 100 persons


## 6. Separate and Validate Reported Totals

Separate the reported total from the sector records and verify that it equals the sum of the individual sectors.

In [6]:
inpatients_by_sector = inpatients_counts[
    inpatients_counts["Sector"] != "Total"
].copy()

inpatients_total = inpatients_counts[
    inpatients_counts["Sector"] == "Total"
].copy()

calculated_total = inpatients_by_sector[inpatient_year_columns].sum()

reported_total = (
    inpatients_total[inpatient_year_columns].iloc[0]
)

total_validation = pd.DataFrame({
    "Calculated sector sum": calculated_total,
    "Reported total": reported_total,
    "Matches": calculated_total == reported_total
})

print("Sector rows:", inpatients_by_sector.shape)
print("Total rows:", inpatients_total.shape)

display(total_validation)

Sector rows: (3, 6)
Total rows: (1, 6)


,Calculated sector sum,Reported total,Matches
2020G,2846773,2846773,True
2021G,3100752,3100752,True
2022G,2702460,2702460,True
2023G,3831824,3831824,True
2024G,3630334,3630334,True


## 7. Reshape Inpatient Sector Data

Transform the inpatient sector data from wide format to analysis-ready long format.

In [7]:
inpatients_by_sector_long = inpatients_by_sector.melt(
    id_vars="Sector",
    value_vars=inpatient_year_columns,
    var_name="year",
    value_name="inpatient_count"
).rename(columns={
    "Sector": "sector"
})

inpatients_by_sector_long["year"] = (
    inpatients_by_sector_long["year"]
    .str.replace("G", "", regex=False)
    .astype("Int64")
)

print("Long sector data shape:", inpatients_by_sector_long.shape)

display(inpatients_by_sector_long)

Long sector data shape: (15, 3)


,sector,year,inpatient_count
0,MoH,2020,1173910
1,Other Governmental Sector,2020,553214
2,Private Sector,2020,1119649
3,MoH,2021,1376787
4,Other Governmental Sector,2021,620735
5,Private Sector,2021,1103230
6,MoH,2022,1389950
7,Other Governmental Sector,2022,544306
8,Private Sector,2022,768204
9,MoH,2023,1418237


## 8. Reshape Remaining Inpatient Metrics

Transform the reported inpatient totals and admissions rates into long format.

In [8]:
inpatients_total_long = inpatients_total.melt(
    value_vars=inpatient_year_columns,
    var_name="year",
    value_name="total_inpatients"
)

inpatients_rate_long = inpatients_rate.melt(
    value_vars=inpatient_year_columns,
    var_name="year",
    value_name="admissions_per_100"
)

for dataframe in [inpatients_total_long, inpatients_rate_long]:
    dataframe["year"] = (
        dataframe["year"]
        .str.replace("G", "", regex=False)
        .astype("Int64")
    )

print("Total inpatient data shape:", inpatients_total_long.shape)
print("Admission rate data shape:", inpatients_rate_long.shape)

display(inpatients_total_long)
display(inpatients_rate_long)

Total inpatient data shape: (5, 2)
Admission rate data shape: (5, 2)


,year,total_inpatients
0,2020,2846773
1,2021,3100752
2,2022,2702460
3,2023,3831824
4,2024,3630334


,year,admissions_per_100
0,2020,9.0
1,2021,10.1
2,2022,8.4
3,2023,11.3
4,2024,10.3


## 9. Reshape Workforce Data

Transform the workforce dataset from wide format to analysis-ready long format.

In [9]:
workforce_long = workforce_clean.melt(
    id_vars=["Category", "Nationality"],
    value_vars=workforce_year_columns,
    var_name="year",
    value_name="workforce_count"
).rename(columns={
    "Category": "category",
    "Nationality": "nationality"
})

workforce_long["year"] = (
    workforce_long["year"]
    .str.replace("G", "", regex=False)
    .astype("Int64")
)

workforce_long["workforce_count"] = (
    workforce_long["workforce_count"].astype("Int64")
)

print("Long workforce data shape:", workforce_long.shape)

display(workforce_long.head(12))

Long workforce data shape: (120, 4)


,category,nationality,year,workforce_count
0,Physicians,Saudi,2020,4307
1,Physicians,Non-Saudi,2020,7044
2,Physicians,Total,2020,11351
3,Dentists,Saudi,2020,2968
4,Dentists,Non-Saudi,2020,75
5,Dentists,Total,2020,3043
6,Total Physicians and Dentists,Saudi,2020,7275
7,Total Physicians and Dentists,Non-Saudi,2020,7119
8,Total Physicians and Dentists,Total,2020,14394
9,Nurses,Saudi,2020,15809


## 10. Validate Workforce Totals

Verify that the reported workforce total equals the sum of Saudi and Non-Saudi workers for every category and year.

In [10]:
workforce_total_validation = (
    workforce_long
    .pivot(
        index=["category", "year"],
        columns="nationality",
        values="workforce_count"
    )
    .reset_index()
)

workforce_total_validation.columns.name = None

workforce_total_validation["calculated_total"] = (
    workforce_total_validation["Saudi"]
    + workforce_total_validation["Non-Saudi"]
)

workforce_total_validation["matches"] = (
    workforce_total_validation["calculated_total"]
    == workforce_total_validation["Total"]
)

print(
    "All workforce totals match:",
    workforce_total_validation["matches"].all()
)

display(workforce_total_validation.head(12))

All workforce totals match: True


,category,year,Non-Saudi,Saudi,Total,calculated_total,matches
0,Allied health Personnel,2020,145,13914,14059,14059,True
1,Allied health Personnel,2021,72,11321,11393,11393,True
2,Allied health Personnel,2022,50,14001,14051,14051,True
3,Allied health Personnel,2023,99,13226,13325,13325,True
4,Allied health Personnel,2024,84,12310,12394,12394,True
5,Dentists,2020,75,2968,3043,3043,True
6,Dentists,2021,13,2510,2523,2523,True
7,Dentists,2022,12,2723,2735,2735,True
8,Dentists,2023,32,2275,2307,2307,True
9,Dentists,2024,20,2388,2408,2408,True


## 11. Separate Workforce Detail and Aggregates

Separate detailed workforce records from reported nationality totals and combined professional categories.

In [11]:
combined_category_names = [
    "Total Physicians and Dentists",
    "Total Nurses and Midwives"
]

is_combined_category = workforce_long["category"].isin(
    combined_category_names
)

is_nationality_total = workforce_long["nationality"] == "Total"

workforce_detail = workforce_long[
    ~is_combined_category & ~is_nationality_total
].copy()

workforce_category_totals = workforce_long[
    ~is_combined_category & is_nationality_total
].copy()

workforce_combined_categories = workforce_long[
    is_combined_category
].copy()

split_row_count = (
    len(workforce_detail)
    + len(workforce_category_totals)
    + len(workforce_combined_categories)
)

print("Workforce detail rows:", workforce_detail.shape)
print("Category total rows:", workforce_category_totals.shape)
print("Combined category rows:", workforce_combined_categories.shape)

print("\nOriginal row count:", len(workforce_long))
print("Split row count:", split_row_count)

display(workforce_detail.head(12))

Workforce detail rows: (60, 4)
Category total rows: (30, 4)
Combined category rows: (30, 4)

Original row count: 120
Split row count: 120


,category,nationality,year,workforce_count
0,Physicians,Saudi,2020,4307
1,Physicians,Non-Saudi,2020,7044
3,Dentists,Saudi,2020,2968
4,Dentists,Non-Saudi,2020,75
9,Nurses,Saudi,2020,15809
10,Nurses,Non-Saudi,2020,2360
12,Midwives,Saudi,2020,204
13,Midwives,Non-Saudi,2020,24
18,Pharmacists,Saudi,2020,564
19,Pharmacists,Non-Saudi,2020,3


## 12. Validate Combined Workforce Categories

Verify that the combined professional categories equal the sum of their component categories.

In [12]:
workforce_combined_validation = (
    workforce_long
    .pivot(
        index=["nationality", "year"],
        columns="category",
        values="workforce_count"
    )
    .reset_index()
)

workforce_combined_validation.columns.name = None

workforce_combined_validation["calculated_physicians_dentists"] = (
    workforce_combined_validation["Physicians"]
    + workforce_combined_validation["Dentists"]
)

workforce_combined_validation["physicians_dentists_match"] = (
    workforce_combined_validation["calculated_physicians_dentists"]
    == workforce_combined_validation["Total Physicians and Dentists"]
)

workforce_combined_validation["calculated_nurses_midwives"] = (
    workforce_combined_validation["Nurses"]
    + workforce_combined_validation["Midwives"]
)

workforce_combined_validation["nurses_midwives_match"] = (
    workforce_combined_validation["calculated_nurses_midwives"]
    == workforce_combined_validation["Total Nurses and Midwives"]
)

all_combined_totals_match = (
    workforce_combined_validation["physicians_dentists_match"].all()
    and workforce_combined_validation["nurses_midwives_match"].all()
)

print("All combined category totals match:", all_combined_totals_match)

display(
    workforce_combined_validation[
        [
            "nationality",
            "year",
            "calculated_physicians_dentists",
            "Total Physicians and Dentists",
            "physicians_dentists_match",
            "calculated_nurses_midwives",
            "Total Nurses and Midwives",
            "nurses_midwives_match"
        ]
    ]
)

All combined category totals match: True


,nationality,year,calculated_physicians_dentists,Total Physicians and Dentists,physicians_dentists_match,calculated_nurses_midwives,Total Nurses and Midwives,nurses_midwives_match
0,Non-Saudi,2020,7119,7119,True,2384,2384,True
1,Non-Saudi,2021,5340,5340,True,2357,2357,True
2,Non-Saudi,2022,5841,5841,True,2202,2202,True
3,Non-Saudi,2023,4717,4717,True,1960,1960,True
4,Non-Saudi,2024,4458,4458,True,2017,2017,True
5,Saudi,2020,7275,7275,True,16013,16013,True
6,Saudi,2021,5610,5610,True,15877,15877,True
7,Saudi,2022,7612,7612,True,18485,18485,True
8,Saudi,2023,7021,7021,True,17785,17785,True
9,Saudi,2024,7643,7643,True,14398,14398,True


## 13. Final Validation

Validate the cleaned datasets before saving them to the processed data directory.

In [13]:
if not total_validation["Matches"].all():
    raise ValueError(
        "Reported inpatient totals do not match the sector sums."
    )

if not workforce_total_validation["matches"].all():
    raise ValueError(
        "Reported workforce totals do not match Saudi plus Non-Saudi."
    )

if not all_combined_totals_match:
    raise ValueError(
        "Combined workforce categories do not match their components."
    )

if split_row_count != len(workforce_long):
    raise ValueError(
        "Some workforce rows were lost or duplicated during separation."
    )

final_datasets = {
    "inpatients_by_sector": inpatients_by_sector_long,
    "inpatients_total": inpatients_total_long,
    "inpatients_admission_rate": inpatients_rate_long,
    "workforce_detail": workforce_detail,
    "workforce_category_totals": workforce_category_totals,
    "workforce_combined_categories": workforce_combined_categories
}

primary_keys = {
    "inpatients_by_sector": ["sector", "year"],
    "inpatients_total": ["year"],
    "inpatients_admission_rate": ["year"],
    "workforce_detail": ["category", "nationality", "year"],
    "workforce_category_totals": ["category", "year"],
    "workforce_combined_categories": [
        "category",
        "nationality",
        "year"
    ]
}

value_columns = {
    "inpatients_by_sector": "inpatient_count",
    "inpatients_total": "total_inpatients",
    "inpatients_admission_rate": "admissions_per_100",
    "workforce_detail": "workforce_count",
    "workforce_category_totals": "workforce_count",
    "workforce_combined_categories": "workforce_count"
}

for dataset_name, dataframe in final_datasets.items():
    missing_values = dataframe.isna().sum().sum()
    duplicate_rows = dataframe.duplicated().sum()

    duplicate_keys = dataframe.duplicated(
        subset=primary_keys[dataset_name]
    ).sum()

    value_column = value_columns[dataset_name]

    negative_values = (
        dataframe[value_column]
        .lt(0)
        .fillna(False)
        .sum()
    )

    print(
        f"{dataset_name}: "
        f"shape={dataframe.shape}, "
        f"missing={missing_values}, "
        f"duplicate_rows={duplicate_rows}, "
        f"duplicate_keys={duplicate_keys}, "
        f"negative_values={negative_values}"
    )

    if missing_values > 0:
        raise ValueError(
            f"Missing values found in {dataset_name}."
        )

    if duplicate_rows > 0:
        raise ValueError(
            f"Duplicate rows found in {dataset_name}."
        )

    if duplicate_keys > 0:
        raise ValueError(
            f"Duplicate business keys found in {dataset_name}."
        )

    if negative_values > 0:
        raise ValueError(
            f"Negative values found in {dataset_name}."
        )

print("\nAll cleaned datasets passed validation.")

inpatients_by_sector: shape=(15, 3), missing=0, duplicate_rows=0, duplicate_keys=0, negative_values=0
inpatients_total: shape=(5, 2), missing=0, duplicate_rows=0, duplicate_keys=0, negative_values=0
inpatients_admission_rate: shape=(5, 2), missing=0, duplicate_rows=0, duplicate_keys=0, negative_values=0
workforce_detail: shape=(60, 4), missing=0, duplicate_rows=0, duplicate_keys=0, negative_values=0
workforce_category_totals: shape=(30, 4), missing=0, duplicate_rows=0, duplicate_keys=0, negative_values=0
workforce_combined_categories: shape=(30, 4), missing=0, duplicate_rows=0, duplicate_keys=0, negative_values=0

All cleaned datasets passed validation.


## 14. Save Processed Datasets

Save the validated and analysis-ready datasets as CSV files.

In [14]:
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

saved_files = []

for dataset_name, dataframe in final_datasets.items():
    file_path = processed_dir / f"{dataset_name}.csv"

    dataframe.to_csv(
        file_path,
        index=False
    )

    saved_files.append(file_path)

print("Saved processed datasets:")

for file_path in saved_files:
    print("-", file_path)

Saved processed datasets:
- c:\Users\Am132\Projects\healthcare-ai-agent\data\processed\inpatients_by_sector.csv
- c:\Users\Am132\Projects\healthcare-ai-agent\data\processed\inpatients_total.csv
- c:\Users\Am132\Projects\healthcare-ai-agent\data\processed\inpatients_admission_rate.csv
- c:\Users\Am132\Projects\healthcare-ai-agent\data\processed\workforce_detail.csv
- c:\Users\Am132\Projects\healthcare-ai-agent\data\processed\workforce_category_totals.csv
- c:\Users\Am132\Projects\healthcare-ai-agent\data\processed\workforce_combined_categories.csv


## 15. Cleaning Summary

- The original raw datasets were loaded without modification.
- Separate working copies were created for cleaning.
- Leading and trailing whitespace was removed from text labels.
- Missing workforce category labels were filled using the grouped table structure.
- Year columns were automatically detected and converted from strings to numeric values.
- Inpatient counts were separated from the admissions-per-100-person rate.
- Sector records were separated from the reported inpatient total.
- Reported inpatient totals matched the calculated sector sums for all years.
- Workforce nationality totals matched the sum of Saudi and Non-Saudi workers.
- Combined workforce categories matched the sum of their component categories.
- Detailed records were separated from reported totals to prevent double counting.
- All analysis-ready datasets were transformed into long format.
- Six validated CSV datasets were saved in `data/processed`.
- The original files in `data/raw` remain unchanged.